# 📊 تحليل Sprint 6 — الخطوة 1: `core/trade.py` + `core/strategy.py`

---

## 🎯 TL;DR

> **`core/trade.py`**: `SignalType` (BUY/SELL/HOLD) + `TradeStatus` (OPEN/CLOSED) + `Trade` (صفقة بحساب PnL تلقائي).
> **`core/strategy.py`**: `BaseStrategy` — عقد يجبر كل استراتيجية على `generate_signal(index, candles)`.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[BaseStrategy ABC] --> B[name property]
    A --> C[generate_signal]
    D[Trade] --> E[SignalType]
    D --> F[TradeStatus]
    D --> G[pnl calculation]
    C --> E

    style A fill:#bbf,stroke:#333,stroke-width:3px
    style D fill:#bfb,stroke:#333,stroke-width:2px
```

---

## 🔬 التشريح

### 1️⃣ `core/trade.py`

#### `SignalType` — إشارات الاستراتيجية

| القيمة | المعنى |
|--------|--------|
| `BUY` | إشارة شراء |
| `SELL` | إشارة بيع |
| `HOLD` | لا إجراء |

**🎯 ترث من `str, Enum`** → قابلة للمقارنة كنص + التخزين.

#### `TradeStatus` — حالة الصفقة

| القيمة | المعنى |
|--------|--------|
| `OPEN` | مفتوحة (لم تُغلق بعد) |
| `CLOSED` | مغلقة (تم حساب PnL) |

#### `Trade` — كائن الصفقة

| الحقل | Default | الدور |
|-------|---------|-------|
| `pair` | — | الزوج |
| `entry_time` / `entry_price` | — | الدخول |
| `size` | `1.0` | حجم الصفقة |
| `exit_time` / `exit_price` | `None` | الخروج (يُملأ عند الإغلاق) |
| `status` | `OPEN` | الحالة |
| `pnl` | `0.0` | الربح/الخسارة |

**🎯 `close()` — حساب PnL:**

```python
self.pnl = (exit_price - self.entry_price) * self.size
```

**مثال:** دخول `1.0800`، خروج `1.0850`، size=`1.0` → `PnL = 0.0050`.

**🎯 ملاحظة:** `Trade` **ليس** `frozen` — لأن `status` و`exit_*` و`pnl` تتغير عند `close()`.

---

### 2️⃣ `core/strategy.py` — العقد

#### `BaseStrategy` — ABC

| العنصر | الدور |
|--------|-------|
| `@property name` | اسم الاستراتيجية (display) |
| `generate_signal(index, candles)` | إشارة عند شمعة محددة |

**🎯 تريكة `index` + `candles`:** الاستراتيجية ترى **الشموع السابقة فقط** (`candles[:index+1]`) — **منع Look-ahead Bias** (استخدام بيانات المستقبل).

**🎯 لماذا `SignalType` وليس `Trade` مباشرة؟** الفصل بين **الإشارة** (Strategy) و**التنفيذ** (Backtest Engine) — Strategy لا تعرف عن المحفظة.

---

## 🧪 التحقق

```python
trade = Trade(pair='EUR/USD', entry_time=..., entry_price=Decimal('1.0800'))
trade.close(exit_time=..., exit_price=Decimal('1.0850'))
# status = CLOSED, pnl = 0.0050
```

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| `pnl` **غير محسوب** في `__post_init__` | OK — يُحسب فقط عند `close()` |
| `close()` لا يتحقق من `status == OPEN` | يمكن إغلاق صفقة مغلقة (idempotency مفقودة) |
| `size` ثابت `1.0` | لا يدعم أحجام ديناميكية (Sprint 7) |
| لا commission / spread | PnL مثالي — يحتاج خصماً في Sprint 7 |
| `signal` لا يحمل `price` | المحرك يعرف السعر من الشمعة |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Separation** | Signal (Strategy) ≠ Execution (Engine) |
| **Immutability** | Strategy ABC ثابت |
| **Look-ahead Protection** | `candles[:index+1]` ضمنياً |
| **Decimal** | PnL دقيق |
| **Enum** | `SignalType` / `TradeStatus` |
| **Value Object** | `Trade` كيان كامل |

---

## 🗺️ Sprint 6

```mermaid
graph LR
    A["core/trade.py + strategy.py 🆕"] --> B["strategies/sma_crossover.py"]
    B --> C["services/backtest_service.py"]
    C --> D["CLI: forex backtest"]
    D --> E["tests/test_backtest.py"]

    style A fill:#bfb,stroke:#333,stroke-width:3px
    style B fill:#ddd,stroke:#333
```

---

## 📋 الخطوة التالية

> **الخطوة 2**: `strategies/sma_crossover.py` — استراتيجية تقاطع SMA.

**الفكرة:**
- **SMA سريع (10)** + **SMA بطيء (30)**.
- **BUY** عندما يتقاطع السريع **فوق** البطيء.
- **SELL** عندما يتقاطع **تحته**.
- **HOLD** خلاف ذلك.

**السؤال الجوهري:** كيف نمنع **التكرار** (إشارات BUY متتالية)؟

**الجواب:** المحرك يتتبع `position` (in/out) — لا يفتح صفقة جديدة لو صفقة مفتوحة.

In [1]:
from datetime import datetime
from decimal import Decimal
from core.trade import Trade, TradeStatus

trade = Trade(pair='EUR/USD', entry_time=datetime.now(), entry_price=Decimal('1.0800'))
trade.close(exit_time=datetime.now(), exit_price=Decimal('1.0850'))

print('Trade Status:', trade.status.value)
print('Profit/Loss :', trade.pnl)


Trade Status: CLOSED
Profit/Loss : 0.00500


# 📈 تحليل `SMACrossoverStrategy` — Sprint 6، الخطوة 2

---

## 🎯 TL;DR

> **استراتيجية Golden Cross / Death Cross**: إشارة BUY عند تقاطع SMA السريع **فوق** البطيء، وSELL عند تقاطعه **تحته**. تعتمد على **شمعة واحدة فقط** (`index`) — لا Lookahead Bias.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[BaseStrategy ABC] --> B[SMACrossoverStrategy]
    B --> C[SMA fast]
    B --> D[SMA slow]
    B --> E[generate_signal]
    E --> F{prev vs curr}
    F -->|Golden| G[BUY]
    F -->|Death| H[SELL]
    F -->|None| I[HOLD]

    style B fill:#bfb,stroke:#333,stroke-width:3px
    style G fill:#bbf,stroke:#333
    style H fill:#fdd,stroke:#333
    style I fill:#ddd,stroke:#333
```

---

## 🔬 التشريح

### 1️⃣ `__init__` — Validation + Composition

```python
if fast_period >= slow_period:
    raise ValidationError(...)
```

**Fail Early**: فترة سريعة ≥ بطيئة → **مستحيل منطقياً** (لا تقاطع).

```python
self._fast_sma = SMA(period=fast_period)
self._slow_sma = SMA(period=slow_period)
```

**Composition**: مؤشران `SMA` (من Sprint 3) — **إعادة استخدام** لا تكرار.

---

### 2️⃣ `name` — property

```python
return f"SMA_Crossover({self.fast_period}/{self.slow_period})"
```

**مثال:** `"SMA_Crossover(10/30)"` — يظهر في تقارير الـ Backtest.

---

### 3️⃣ `generate_signal` — **قلب الاستراتيجية**

```mermaid
graph TD
    A[index] --> B{index < slow_period?}
    B -->|Yes| C[HOLD]
    B -->|No| D["sub_candles = candles[:index+1]"]
    D --> E[calc fast_series + slow_series]
    E --> F[extract curr/prev]
    F --> G{None in values?}
    G -->|Yes| H[HOLD]
    G -->|No| I{prev_fast <= prev_slow<br/>AND curr_fast > curr_slow?}
    I -->|Yes| J[BUY]
    I -->|No| K{prev_fast >= prev_slow<br/>AND curr_fast < curr_slow?}
    K -->|Yes| L[SELL]
    K -->|No| M[HOLD]

    style J fill:#bbf,stroke:#333,stroke-width:2px
    style L fill:#fdd,stroke:#333,stroke-width:2px
    style C fill:#ddd,stroke:#333
    style H fill:#ddd,stroke:#333
    style M fill:#ddd,stroke:#333
```

#### الخطوات:

| # | الخطوة | السبب |
|---|--------|-------|
| 1 | `if index < slow_period: HOLD` | لا يكفي بيانات للمتوسط البطيء |
| 2 | `sub_candles = candles[:index+1]` | **منع Lookahead Bias** |
| 3 | `calculate` للمؤشرين | الحصول على السلسلة |
| 4 | `[-1]` و `[-2]` | الحالية + السابقة |
| 5 | `if None in (...)`: HOLD | حماية من قيم غير محسوبة |
| 6 | **Golden Cross** | `prev_fast <= prev_slow` **AND** `curr_fast > curr_slow` |
| 7 | **Death Cross** | `prev_fast >= prev_slow` **AND** `curr_fast < curr_slow` |
| 8 | خلاف ذلك | HOLD |

---

## 🎯 التريكات الأساسية

### 1. منع Lookahead Bias

```python
sub_candles = candles[: index + 1]
```

**المشكلة بدونها:** لو حسبنا SMA على **كل الشموع**، القيم في `index` ستتضمن شموع المستقبل → إشارات "سحرية" غير واقعية.

**الحل:** قطع الشموع عند `index` فقط — الاستراتيجية ترى **الماضي والحاضر، ليس المستقبل**.

**🎯 الأداء:** O(N) لكل إشارة → O(N²) إجمالاً. **يمكن تحسينه** بـ rolling window في Sprint 7 إذا لزم.

### 2. شرط التقاطع المزدوج

```python
if prev_fast <= prev_slow and curr_fast > curr_slow:
```

**لماذا `<=` وليس `<`؟**

| السيناريو | `prev_fast == prev_slow` |
|-----------|--------------------------|
| Golden Cross مع `<` | ❌ يفوت التقاطع |
| Golden Cross مع `<=` | ✅ يكتشفه |

**🎯 ضروري:** القيم قد تكون **متساوية** في لحظة معينة (لا تقاطع صارم)، ثم تنفصل — يجب اعتبارها تقاطعاً.

### 3. `None` Check قبل المقارنة

```python
if None in (curr_fast, curr_slow, prev_fast, prev_slow):
    return SignalType.HOLD
```

**السبب:** أول `slow_period-1` قيمة = `None` في SMA. لو مررناها للمقارنة → `TypeError`.

**🎯 تريكة:** `None in tuple` — فحص واحد بدل 4 شروط `or`.

### 4. `assert` بعد `None` Check

```python
assert curr_fast is not None and curr_slow is not None
```

**السبب:** Type Narrowing — `mypy` لا يفهم `None in tuple` كنتيجة. `assert` يخبر المحلل بأنها `Decimal` بعد الفحص.

---

## 🧪 التحقق

**السيناريو:** 15 شمعة ثابتة (1.05) → هبوط (1.04 → 1.02) → صعود حاد (1.035 → 1.095).

| اللحظة | Fast SMA (3) | Slow SMA (7) | التقاطع |
|--------|--------------|--------------|---------|
| قبل الهبوط | 1.05 | 1.05 | HOLD |
| أثناء الهبوط | < slow | > fast | HOLD (لا تقاطع) |
| **بعد الصعود** | **> slow** | **< fast** | **BUY** ✅ |

**المتوقع:** إشارة BUY عند مؤشر الشمعة الأخيرة تقريباً.

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| `calculate` كامل السلسلة لكل `index` | O(N²) — بطيء لـ 10K شمعة |
| لا `position` tracking | كل استدعاء مستقل (المحرك يتابع) |
| `assert` في production | يُعطَّل بـ `python -O` — قد يمرر `None` |
| HOLD عند `None` | صحيح — لكن قد يفوت تقاطعاً لو SMA لم يكتمل بعد |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Lookahead Prevention** | `candles[:index+1]` |
| **Composition** | مؤشران `SMA` |
| **Fail Early** | `fast_period >= slow_period` |
| **Fail Soft** | `None` → `HOLD` |
| **Declarative** | شرط تقاطع واضح |
| **Reusability** | `SMA` من Sprint 3 |
| **Immutability** | الاستراتيجية لا تعدل الشموع |

---

## 🗺️ Sprint 6

```mermaid
graph LR
    A["core/trade.py + strategy.py ✅"] --> B["strategies/sma_crossover.py 🆕"]
    B --> C["services/backtest_service.py"]
    C --> D["CLI: forex backtest"]
    D --> E["tests"]

    style B fill:#bfb,stroke:#333,stroke-width:3px
```

---

## 📋 الخطوة التالية

> **الخطوة 3**: `services/backtest_service.py` — محرك المحاكاة.

**المكونات:**
1. **`BacktestResult`**: `total_trades`, `win_rate`, `roi`, `max_drawdown`.
2. **`BacktestService.run(strategy, candles, initial_capital)`**:
   - Loop على الشموع
   - `generate_signal` لكل شمعة
   - BUY → فتح `Trade` + خصم رأس المال
   - SELL → إغلاق `Trade` + إضافة PnL
   - حساب Metrics في النهاية

**السؤال الجوهري:** كيف نُغلق الصفقة المفتوحة في نهاية الباكتست؟

**الجواب:** **Force Close** عند آخر شمعة — `trade.close(last_candle)` — لتجنب "صفقة معلقة" في الإحصائيات.

In [2]:
from datetime import datetime, timedelta
from decimal import Decimal
from core.candle import Candle
from core.trade import SignalType
from strategies.sma_crossover import SMACrossoverStrategy

# توليد بيانات مصطنعة لمحاكاة هبوط ثم صعود حاد يؤدي إلى تقاطع صاعد (Golden Cross)
prices = [1.0500] * 15 + [1.0400, 1.0350, 1.0300, 1.0250, 1.0200] + [1.0350, 1.0550, 1.0750, 1.0950]
base_time = datetime.now() - timedelta(days=len(prices))

candles = [
    Candle(
        pair='EUR/USD',
        interval='1day',
        timestamp=base_time + timedelta(days=i),
        open=Decimal(str(p)),
        high=Decimal(str(p + 0.0010)),
        low=Decimal(str(p - 0.0010)),
        close=Decimal(str(p)),
    )
    for i, p in enumerate(prices)
]

strategy = SMACrossoverStrategy(fast_period=3, slow_period=7)

signals = []
for idx in range(len(candles)):
    sig = strategy.generate_signal(idx, candles)
    if sig != SignalType.HOLD:
        signals.append((idx, candles[idx].timestamp.strftime('%Y-%m-%d'), sig.value, candles[idx].close))

print(f'Strategy Name: {strategy.name}')
print('Generated Signals:', signals)

Strategy Name: SMA_Crossover(3/7)
Generated Signals: [(15, '2026-10-28', 'SELL', Decimal('1.04')), (21, '2026-11-03', 'BUY', Decimal('1.055'))]


# ⚙️ تحليل `BacktestService` — محرك المحاكاة (Sprint 6، الخطوة 3)

---

## 🎯 TL;DR

> **محرك محاكاة** يمر على الشموع زمنياً، يفتح صفقة عند BUY، يُغلقها عند SELL، ويجبر الإغلاق في النهاية. يحسب **6 مقاييس أداء**: Total Trades, Win Rate, ROI, Max Drawdown, Profit Factor, PnL.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[BacktestService.run] --> B[get_candles]
    B --> C[sorted ascending]
    C --> D[Loop on candles]
    D --> E{signal?}
    E -->|BUY + no open| F[open Trade]
    E -->|SELL + open| G[close + capital += pnl]
    E -->|HOLD| H[skip]
    F --> I[update drawdown]
    G --> I
    I --> D
    D --> J{open at end?}
    J -->|Yes| K[force close]
    J -->|No| L[compute metrics]
    K --> L
    L --> M[return BacktestResult]

    style G fill:#bfb,stroke:#333,stroke-width:2px
    style K fill:#f9f,stroke:#333,stroke-width:2px
    style M fill:#bdf,stroke:#333,stroke-width:2px
```

---

## 🔬 التشريح

### 1️⃣ `BacktestResult` — Value Object (14 حقل)

| الفئة | الحقول |
|-------|--------|
| **السياق** | `pair`, `interval`, `strategy_name` |
| **رأس المال** | `initial_capital`, `final_capital` |
| **الصفقات** | `total_trades`, `winning_trades`, `losing_trades` |
| **المقاييس** | `win_rate`, `total_pnl`, `roi_percent`, `max_drawdown_percent`, `profit_factor` |
| **التفاصيل** | `trades: list[Trade]` |

**🎯 `to_dict()`:** تحويل لـ `float` + `round(2)` — مناسب للـ JSON والـ CLI.

---

### 2️⃣ `run` — 7 مراحل

#### المرحلة 1: جلب + فرز

```python
candles = storage.get_candles(..., limit=limit)
if not candles: raise ValidationError(...)
candles_sorted = sorted(candles, key=lambda c: c.timestamp)
```

**🎯 `ValidationError` مع hint:** "استخدم `forex fetch` أولاً" — **UX ممتاز**.

**🎯 `sorted`:** `get_candles` يُرجع DESC → نحتاج ASC.

#### المرحلة 2: تهيئة الحالة

```python
capital = initial_capital
peak_capital = initial_capital
max_drawdown = Decimal("0.0")
open_trade = None
executed_trades = []
```

| المتغير | الدور |
|---------|-------|
| `capital` | الرصيد الحالي (يتغير مع PnL) |
| `peak_capital` | أعلى رصيد تاريخي (للـ Drawdown) |
| `max_drawdown` | أقصى هبوط مسجل |
| `open_trade` | الصفقة المفتوحة (أو `None`) |

#### المرحلة 3: Loop التداول

```mermaid
graph TD
    A[for each candle] --> B[signal = strategy.generate_signal]
    B --> C{BUY and no open?}
    C -->|Yes| D[open Trade at close]
    C -->|No| E{SELL and open?}
    E -->|Yes| F[close + capital += pnl]
    E -->|No| G[skip]
    D --> H[update drawdown]
    F --> H
    G --> H
    H --> A

    style D fill:#bbf,stroke:#333
    style F fill:#bfb,stroke:#333
```

**🎯 تريكة `if BUY and open_trade is None`:** منع فتح صفقتين متتاليتين — **Idempotent opening**.

**🎯 تريكة `if SELL and open_trade is not None`:** منع البيع بدون صفقة — **Idempotent closing**.

**🎯 البيع عند السعر الحالي:** `candle.close` — نفس آلية الدخول (سعر الإغلاق).

#### المرحلة 4: حساب Drawdown (داخل الحلقة)

```python
if capital > peak_capital:
    peak_capital = capital
drawdown = (peak_capital - capital) / peak_capital * 100
if drawdown > max_drawdown:
    max_drawdown = drawdown
```

**الفكرة:**
- `peak_capital` يرتفع فقط.
- `drawdown` = نسبة الهبوط من القمة.
- `max_drawdown` = أكبر هبوط مسجل.

**مثال:**
| الخطوة | capital | peak | drawdown |
|--------|---------|------|----------|
| 1 | 10000 | 10000 | 0% |
| 2 | 10500 | 10500 | 0% |
| 3 | 10200 | 10500 | 2.86% ← |
| 4 | 10700 | 10700 | 0% |
| 5 | 9800 | 10700 | 8.41% ← max |

#### المرحلة 5: Force Close

```python
if open_trade is not None:
    open_trade.close(exit_time=last.timestamp, exit_price=last.close)
    capital += open_trade.pnl
    executed_trades.append(open_trade)
```

**🎯 لماذا إجباري؟**
- لو انتهت الشموع بصفقة مفتوحة، **PnL غير محسوب**.
- الإحصائيات ستكون خاطئة (عدد صفقات ناقص).

#### المرحلة 6: مقاييس الأداء

| المقياس | الصيغة | ملاحظة |
|---------|--------|--------|
| `win_rate` | `winning / total × 100` | `0` لو مفيش صفقات |
| `total_pnl` | `capital - initial` | Decimal دقيق |
| `roi_percent` | `(pnl / initial) × 100` | float للعرض |
| `profit_factor` | `gross_profit / gross_loss` | 3 حالات! |

**🎯 تريكة `profit_factor` — 3 حالات:**

```python
if gross_loss > 0:
    profit_factor = gross_profit / gross_loss
elif gross_profit > 0:
    profit_factor = float(gross_profit)   # لا خسائر → PF = ∞
else:
    profit_factor = 1.0                   # لا شيء → محايد
```

**الحالات:**

| `gross_loss` | `gross_profit` | `profit_factor` |
|--------------|----------------|-----------------|
| `> 0` | أي قيمة | `profit / loss` |
| `0` | `> 0` | `∞` (تُمثل بـ gross_profit) |
| `0` | `0` | `1.0` (محايد) |

#### المرحلة 7: الإرجاع

`BacktestResult` مع 14 حقل + `trades` الكاملة.

---

## 📊 معنى المقاييس

| المقياس | القيمة الجيدة | المعنى |
|---------|---------------|--------|
| **Win Rate** | > 50% | نسبة صفقات رابحة |
| **ROI** | > 0% | عائد على رأس المال |
| **Max Drawdown** | منخفض | أقصى خسارة مؤقتة |
| **Profit Factor** | > 1.5 | ربح/خسارة إجمالي |

---

## 🧪 التحقق (وصف)

**السيناريو:** 6 شموع هبوط ثم 6 صعود → تقاطعان محتملان.

| المتوقع | القيمة |
|---------|--------|
| Total Trades | 1-2 |
| Win Rate | 0% أو 100% |
| Max Drawdown | > 0% (لو خسر) |
| ROI | موجب/سالب |

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| `trade_size = 10000` | قد يتجاوز رأس المال (`initial=10000`) — **Risk** |
| لا commission/spread | PnL مثالي |
| `size` = رأس المال كامل | لا يوجد position sizing (Sprint 7) |
| لا Stop Loss / Take Profit | ينتظر SELL فقط |
| `max_drawdown` يُحدّث حتى بدون صفقات | صحيح — يراقب المحفظة |
| لا benchmark (Buy & Hold) | صعب تقييم الاستراتيجية |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Sequential Simulation** | Loop زمني دقيق |
| **State Machine** | `open_trade` يتحول `None ↔ Trade` |
| **Force Close** | لا صفقات معلقة |
| **Decimal** | PnL دقيق |
| **Fail Early** | `ValidationError` لو لا شموع |
| **Fail Safe** | `profit_factor` 3 حالات |
| **Reusability** | `BaseStrategy` — أي استراتيجية |
| **Hint UX** | "استخدم `forex fetch`" |

---

## 🗺️ Sprint 6

```mermaid
graph LR
    A["core/trade.py + strategy.py ✅"] --> B["strategies/sma_crossover.py ✅"]
    B --> C["services/backtest_service.py 🆕"]
    C --> D["CLI: forex backtest"]
    D --> E["tests"]

    style C fill:#bfb,stroke:#333,stroke-width:3px
```

---

## 📋 الخطوة التالية

> **الخطوة 4**: `forex backtest` — أمر CLI.

| Option | Default | الدور |
|--------|---------|-------|
| `--pair` | EUR/USD | الزوج |
| `--interval` | 1day | الإطار |
| `--limit` | 100 | عدد الشموع |
| `--capital` | 10000 | رأس المال |
| `--fast` / `--slow` | 10/30 | إعدادات SMA |

**السؤال الجوهري:** كيف نعرض 14 حقل في جدول؟

**الجواب:** جدول من قسمين:
- **Summary** (pair, strategy, capital)
- **Metrics** (trades, win rate, ROI, drawdown, PF)

In [3]:

from datetime import datetime, timedelta
from decimal import Decimal
from core.candle import Candle
from services.backtest_service import BacktestService
from storage.sqlite_storage import SQLiteStorage
from strategies.sma_crossover import SMACrossoverStrategy

storage = SQLiteStorage(db_path=':memory:')

# حفظ شموع تتضمن هبوطاً ثم صعوداً حاداً لتوليد صفقات
prices = [1.0500] * 12 + [1.0300, 1.0200, 1.0100] + [1.0400, 1.0600, 1.0800, 1.0700, 1.0500, 1.0300]
start = datetime.now() - timedelta(days=len(prices))

candles = [
    Candle(
        pair='EUR/USD', interval='1day',
        timestamp=start + timedelta(days=i),
        open=Decimal(str(p)), high=Decimal(str(p + 0.0010)),
        low=Decimal(str(p - 0.0010)), close=Decimal(str(p))
    )
    for i, p in enumerate(prices)
]
storage.save_candles_bulk(candles)

# تشغيل المحاكاة
service = BacktestService(storage=storage)
strategy = SMACrossoverStrategy(fast_period=3, slow_period=6)
result = service.run(strategy=strategy, pair='EUR/USD', limit=50)

print('--- Backtest Summary ---')
print('Strategy      :', result.strategy_name)
print('Total Trades  :', result.total_trades)
print('Win Rate      :', f'{result.win_rate:.1f}%')
print('Total PnL     :', f'${result.total_pnl:.2f}')
print('ROI           :', f'{result.roi_percent:.2f}%')
print('Max Drawdown  :', f'{result.max_drawdown_percent:.2f}%')


[2026-11-06 01:12:20] [INFO] [forex_tracker]: Connected to SQLite database at: :memory:
[2026-11-06 01:12:20] [INFO] [forex_tracker]: Saved 21 candle(s) to SQLite.
[2026-11-06 01:12:20] [INFO] [forex_tracker]: Backtest completed for EUR/USD [SMA_Crossover(3/6)]: Trades=1, WinRate=0.0%, ROI=-3.00%


--- Backtest Summary ---
Strategy      : SMA_Crossover(3/6)
Total Trades  : 1
Win Rate      : 0.0%
Total PnL     : $-300.00
ROI           : -3.00%
Max Drawdown  : 3.00%
